# Learning Unit 5 — Fairness

In this learning unit, we are going to investigate different fairness approaches and techniques to mitigate bias.
We will continue using the *CDC Diabetes Health Indicators* dataset and your predictions from the last Unit. 

During the lecture, the following fairness metrics were introduced: **Group Fairness**, **Equalized Odds (Equal error rates)**, and **Test Fairness (Calibration)**. 
In this exercise we will go over some metrics (**Group Fairness**, **Predictive Parity (PPV / Precision)**, **False Positive Error Rate Balance (FPR)**) again and briefly explain what they exactly capture. 

Afterward, you will freely evaluate the dataset and search for (un)fairness. For this purpose, you do not have to implement these metrics on your own, we offer you already-implemented functions. You don't need to understand how they're implemented in detail, but please don't hesitate to look at them or adjust them in the *fairness_functions.py* file. 

---

## Data preparation

In [32]:
import fairness_functions as ff

**✏️ Task 5.1**
*Load in the dataset into a Pandas dataframe and add your predictions from LU-4 Task 4.4 as a new column "prediction".*

In [33]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [34]:
# Load the dataset
df = pd.read_csv('diabetes_012_health_indicators_BRFSS2015.csv')

# Reproduce the exact same split from learning unit-4 to get correct test indices
X = df.drop(columns=['Diabetes_012'])
y = df['Diabetes_012']
_, X_test, _, _ = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Load predictions csv
preds = pd.read_csv('diabetes_predictions.csv')

# Add prediction column — only test rows get values, rest are NaN
df['prediction'] = pd.NA
df.loc[X_test.index, 'prediction'] = preds['Predicted'].values

# print(df[['Diabetes_012', 'prediction']].tail())
print(f"\nShape: {df.shape}")


Shape: (253680, 23)


In [35]:
df.head(30)

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,prediction
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0,<NA>
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0,<NA>
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0,<NA>
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0,<NA>
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0,<NA>
5,0.0,1.0,1.0,1.0,25.0,1.0,0.0,0.0,1.0,1.0,...,0.0,2.0,0.0,2.0,0.0,1.0,10.0,6.0,8.0,0.0
6,0.0,1.0,0.0,1.0,30.0,1.0,0.0,0.0,0.0,0.0,...,0.0,3.0,0.0,14.0,0.0,0.0,9.0,6.0,7.0,<NA>
7,0.0,1.0,1.0,1.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,3.0,0.0,0.0,1.0,0.0,11.0,4.0,4.0,<NA>
8,2.0,1.0,1.0,1.0,30.0,1.0,0.0,1.0,0.0,1.0,...,0.0,5.0,30.0,30.0,1.0,0.0,9.0,5.0,1.0,<NA>
9,0.0,0.0,0.0,1.0,24.0,0.0,0.0,0.0,0.0,0.0,...,0.0,2.0,0.0,0.0,0.0,1.0,8.0,4.0,3.0,<NA>


*The target variable 'Diabetes_012' has three levels, 0 = no diabetes, 1 = diabetes typ I and 2 = diabetes typ II.*
<br> *However, in order to use the fairness metrics we discussed in the lecture we need a binary target variable.*

**✏️ Task 5.2**
*Combine the values of 1 and 2 to receive a binary target variable for diabetes, do this to your prediction variable as well.*
<br> <br> *Hint: Both columns should only contain the values 0 (no diabetes) and 1 (diabetes) as integers, else you will run into trouble later.*

In [36]:
# Binarize Diabetes_012: combine 1 and 2 into 1
df['Diabetes_012'] = df['Diabetes_012'].replace({2: 1})

# Binarize prediction column as well
df['prediction'] = df['prediction'].replace({2: 1})

# Verify both columns only contain 0 and 1
print("Diabetes_012 unique values:", df['Diabetes_012'].unique())
print("prediction unique values:", df['prediction'].dropna().unique())

# Ensure integers
df['Diabetes_012'] = df['Diabetes_012'].astype(int)
df['prediction'] = pd.array(df['prediction'], dtype=pd.Int64Dtype())
# df['prediction'] has NaN values for the train rows and regular int in pandas cannot hold NaN. 
# So pd.Int64Dtype() is used which is pandas' nullable integer type that supports NaN.

print("\nDone. Value counts for Diabetes_012:")
print(df['Diabetes_012'].value_counts())
print("\nValue counts for prediction:")
print(df['prediction'].value_counts())

Diabetes_012 unique values: [0. 1.]
prediction unique values: [0.0 1]

Done. Value counts for Diabetes_012:
Diabetes_012
0    213703
1     39977
Name: count, dtype: int64

Value counts for prediction:
prediction
0    47944
1     2792
Name: count, dtype: Int64


**✏️ Task 5.3**
*Briefly explain why this could be a bad idea if we were to group this variable in reality?*

*Your answer:*


---

Combining Type I and Type II diabetes into a single binary variable oversimplifies the reality of these two conditions. While both are forms of diabetes, they are fundamentally different diseases. They have different causes, different risk factors, different treatments, and affect different populations. By merging them into one label, we lose this important distinction. A model trained on this binary variable would treat a Type I patient the same as a Type II patient, which could lead to misleading predictions and potentially harmful decisions, especially in a healthcare context.

## Fairness

#### Predictive Parity (PPV / Precision)

<b> Members of each group have the same Positive Predictive Value (PPV) — the probability of a subject with Positive Predicted Value to truly belong to the positive class. </b>

The PPV is calculated as:  $\frac{True Positives}{True Positives + False Positives}$

<p align="left">
    <img src="4x4matrix_clear_ppv.png" alt="drawing" width="500"/>
</p>

<br> <br>
A high PPV indicates that we can be sure that a positive prediction is true. Ideally this value is 1, then we can be certain that the (positive!) prediction is true.

In our example, we would want to analyze whether women or men are less likely to truly belong to the positive class and whether there is a significant difference between these two groups.

In [37]:
# The following function returns the probability for subjects of a group (arg3) to truly belong to the positive class (Diabetes / 1). 
# E.g. If you take a random subject from your group that is predicted to belong to the positive class it will have the given probability to truly belong to the positive class.

# arg1 = dataframe, arg2 = group (column name), arg3 = group name (value name), arg4 = prediction (column name) , arg5 = true label (column name)
# e.g. ff.predictive_parity(df, "Sex", 0, "prediction", "Diabetes_binary")

# Filter to test rows only (where prediction is not NaN)
df_test = df.dropna(subset=['prediction']).copy()
df_test['prediction'] = df_test['prediction'].astype(int)

# Predictive Parity for Sex; Sex: 0 = female, 1 = male
ppv_female = ff.predictive_parity(df_test, "Sex", 0, "prediction", "Diabetes_012")
ppv_male = ff.predictive_parity(df_test, "Sex", 1, "prediction", "Diabetes_012")

print(f"PPV Female (Sex=0): {ppv_female}")
print(f"PPV Male   (Sex=1): {ppv_male}")

PPV Female (Sex=0): 0.47891963109354413
PPV Male   (Sex=1): 0.46153846153846156


#### False Positive Error Rate Balance (FPR)

<b> Members of each group have the same False Positive Rate (FPR) — the probability of a subject in the negative class to have a positive predicted value. </b>

The FPR is calculated as:  $\frac{False Positives}{False Positives + True Negatives}$

<p align="left">
    <img src="4x4matrix_clear_fpr.png" alt="drawing" width="500"/>
</p>

The FPR gives insight into the balance between True Negatives and False Positives, If the value is high most of the true negatives are predicted as true, if it is low the algorithm detects most of the true negatives as actually negative.
<br>

In our example, we would want to analyze if any group is disadvantaged by having a higher FPR than the other, thus predicting it more often to be prone to Diabetes even though they are not prone.

In [38]:
# The following function returns the probability for subjects of a group (arg3) to actually belong to the negative class even though they were predicted positive. 

# arg1 = dataframe, arg2 = group (column name), arg3 = group name (value name), arg4 = prediction (column name) , arg5 = true label (column name)
# e.g. ff.fp_error_rate_balance(df, "Sex", 0, "prediction", "Diabetes_binary")

# False Positive Error Rate Balance for Sex
fpr_female = ff.fp_error_rate_balance(df_test, "Sex", 0, "prediction", "Diabetes_012")
fpr_male = ff.fp_error_rate_balance(df_test, "Sex", 1, "prediction", "Diabetes_012")

print(f"FPR Female (Sex=0): {fpr_female}")
print(f"FPR Male   (Sex=1): {fpr_male}")

FPR Female (Sex=0): 0.03293089092422981
FPR Male   (Sex=1): 0.036643341701832165


#### Group Fairness
<b>Members of each group need to have the same probability of being assigned to the positively predicted class.</b>
<br><br>
In our case we will define the positive class as being predicted with diabetes. <br>
For example, if we investigate the group 'Sex' then both groups, protected and unprotected should ideally have the same probability to receive a diabetes prediction. Mathematically this is stated as followed:

$P(DiabetesPrediction =1 \mid Sex = female) == P(DiabetesPrediction =1 \mid Sex = male)$  

Here is how you can calculate this dependent probability for <b>one</b> group:

$P(DiabetesPrediction =1 \mid Sex = female) = \frac{P(DiabetesPrediction =1  \cap  Sex = female)}{P(Sex = female)}$

The $ \cap $ (Intersection) of Diabetes = 1 and Sex = female contains all entries where individuums of female sex are predicted to have diabetes.

If you calculate this probability for both groups you can then compare them and check if Group Fairness is given or not.

In [39]:
# The following function returns the probability of a given group (arg3) to be assigned to the positively predicted class (arg5). 

# arg1 = dataframe, arg2 = group (column name), arg3 = groupname (value name), arg4 = prediction (column name), arg5 = positive class (value name) 
# e.g. ff.group_fairness(df, "Sex", 0, "prediction", 1)

# Group Fairness for Sex
gf_female = ff.group_fairness(df_test, "Sex", 0, "prediction", 1)
gf_male = ff.group_fairness(df_test, "Sex", 1, "prediction", 1)

print(f"Group Fairness Female (Sex=0): {gf_female}")
print(f"Group Fairness Male   (Sex=1): {gf_male}")

Group Fairness Female (Sex=0): 0.054
Group Fairness Male   (Sex=1): 0.057


**✏️ Task 5.4**
*Now, use the metrics above to explore whether the protected variables in this dataset are distributed fairly. What are the most interesting findings?*

*Your answer:*


---

Looking at the 3 metrics for the protected variable Sex, the model appears to be fairly balanced between males and females.
The PPV values are very close - 0.479 for females and 0.462 for males - meaning both groups have a roughly equal chance that a positive diabetes prediction is actually correct. The difference of about 1.7% is not significant enough to raise serious fairness concerns.
The FPR values are also similar - 0.033 for females and 0.037 for males - meaning males are only very slightly more likely to be falsely predicted as diabetic when they are not. The difference is small and does not indicate a strong bias.
Finally, group fairness shows that 5.4% of females and 5.7% of males are predicted to have diabetes, which is nearly identical.
Overall, the most interesting finding is that despite Sex being a biological factor that can influence diabetes risk, the model does not appear to treat the two groups in a significantly different way across any of the 3 fairness metrics. This suggests the model is relatively fair with respect to Sex as a protected variable.

## Bias Mitigation

In the lecture we introduced different bias mitigation strategies. We now want to explore two different methods that are often used by machine learning practitioners to mitigate bias.

<b> 1. Fairness through unawarness: </b> <br>
This relatively straight forward method involves removing all protected attributes from the dataset to prevent the machine learning algorithm from even considering them in the prediction.

<b> 2. Sampling: </b> <br>
Is also quite simple, you look at the representation of all protected attributes. If any attribute is unequally distributed you equalize the number of samples between the group. This can be done in two ways
* either by sampling an underrepresented group multiple times 
* or downsampling an overrepresented group using sampling rules of your choice

**✏️ Task 5.5**
*Select the Bias Mitigation strategy that you believe best fits our context and apply it to your data. Briefly explain your choice of strategy.*

In [40]:
# Drop protected attributes from the full dataset before retraining
df_mitigated = df.drop(columns=['Sex', 'Age'])

print(df_mitigated.columns.tolist())

['Diabetes_012', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Education', 'Income', 'prediction']


*Your answer:*


Since our fairness analysis showed that Sex does not cause significant bias in our model, the issue is not about unequal group representation but rather about the model potentially using protected attributes to make predictions. Removing them entirely is a straightforward and appropriate solution for our context.

**✏️ Task 5.6**
*Train a new model with your newly bias mitigated dataset.*

In [41]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [42]:
# Features and target from mitigated dataset
X_mitigated = df_mitigated.drop(columns=['Diabetes_012', 'prediction'])
y_mitigated = df_mitigated['Diabetes_012']

# Same split as learning unit -4
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_mitigated, y_mitigated, test_size=0.2, random_state=42, stratify=y_mitigated
)

In [43]:
# Same model as learning unit -4 best model
knn_mitigated = KNeighborsClassifier(n_neighbors=11, weights='distance', n_jobs=-1)
knn_mitigated.fit(X_train_m, y_train_m)
knn_mitigated_pred = knn_mitigated.predict(X_test_m)

print(f"Mitigated Model Accuracy: {accuracy_score(y_test_m, knn_mitigated_pred):.4f}")

Mitigated Model Accuracy: 0.8323


**✏️ Task 5.7**
*Use the fairness metrics mentioned earlier on your new model and compare them with your previous results. Is your new model fairer than your previous one?*

In [44]:
# test dataframe for mitigated model
# Build test dataframe for mitigated model
df_test_m = X_test_m.copy()
df_test_m['Diabetes_012'] = y_test_m.values
df_test_m['prediction'] = knn_mitigated_pred
df_test_m['Sex'] = df.loc[X_test_m.index, 'Sex'].values

In [45]:
# Predictive Parity
ppv_female_m = ff.predictive_parity(df_test_m, "Sex", 0, "prediction", "Diabetes_012")
ppv_male_m = ff.predictive_parity(df_test_m, "Sex", 1, "prediction", "Diabetes_012")

# False Positive Error Rate Balance
fpr_female_m = ff.fp_error_rate_balance(df_test_m, "Sex", 0, "prediction", "Diabetes_012")
fpr_male_m = ff.fp_error_rate_balance(df_test_m, "Sex", 1, "prediction", "Diabetes_012")

# Group Fairness
gf_female_m = ff.group_fairness(df_test_m, "Sex", 0, "prediction", 1)
gf_male_m = ff.group_fairness(df_test_m, "Sex", 1, "prediction", 1)

In [46]:
print(f"PPV Female: {ppv_female_m} | PPV Male: {ppv_male_m}")
print(f"FPR Female: {fpr_female_m} | FPR Male: {fpr_male_m}")
print(f"GF  Female: {gf_female_m} | GF  Male: {gf_male_m}")

PPV Female: 0.42773333333333335 | PPV Male: 0.4166089965397924
FPR Female: 0.04467110741049126 | FPR Male: 0.045029645852251485
GF  Female: 0.067 | GF  Male: 0.064


*Your answer:*


The mitigated model shows mixed results. On one hand, the FPR is now almost identical between females and males (0.045 vs 0.045), which is an improvement over the original model where there was a small gap. Similarly, Group Fairness became slightly more balanced, with females now having a marginally higher prediction rate than males, reversing the small gap from before.
On the other hand, the PPV dropped for both groups compared to the original model, meaning positive predictions are now slightly less reliable overall. Additionally, the overall accuracy dropped from 0.8374 to 0.8323.
Overall, the mitigated model is marginally fairer in terms of FPR balance, but the differences in both models are so small that neither can be considered significantly unfair or fair with respect to Sex. The trade-off is a slight loss in accuracy and precision for a minimal gain in fairness.

If you are further interested in Bias Mitigation strategies there are also more advanced methods, the AI Fairness 360 library has a variety of them implemented: https://github.com/Trusted-AI/AIF360?tab=readme-ov-file

## 📝 Feedback
We are interested in your feedback in order to improve this course. We will read all of your feedback and evaluate it. What you share may have a direct impact on the rest of the course or future iterations of it.

Write down your feedback on the lecture, the exercises, or the assignments in the Markdown cell below. Furthermore, please note the approximate time it took you to complete the assignment. You may also write about your insights, what you found interesting, or questions that you have.